<a href="https://colab.research.google.com/github/husthorng/Backpropagation_NN/blob/main/GA16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [78]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [84]:
import numpy as np
#下載W1W2.npz
#https://github.com/husthorng/Backpropagation_NN/blob/main/W1W2.npz
w=np.load("/content/drive/MyDrive/W1W2.npz")
W1 = w['W1']
W2 = w['W2']
max_v=w['max_v']
min_v=w['min_v']
range_V=np.stack((w['max_v'], w['min_v']), axis=0)

# ---------------- Activation ----------------
def logistic(x):
    return 1/(1+np.exp(-x))

def evaluate(pop):

    x = (pop - range_V[1,0:4]) / (range_V[0,0:4]-range_V[1,0:4])
    X = np.hstack((x, np.ones((x.shape[0], 1))))
    h = logistic(X @ W1)
    H = np.hstack((h, np.ones((h.shape[0],1))))
    return logistic(H @ W2)


# ---------------- Parameters ----------------
nvar = 4
npop = 150
gNr = 3000

crossprob = 0.8
mutprob = 0.02
mut_scale = 0.015
p_tour = 0.7

tar_val=range_V[0,:]*1
tar_val[5]=700
CTe=(tar_val-range_V[1,:])/(range_V[0,:]-range_V[1,:])

pop = range_V[1,0:4] + (range_V[0,0:4]-range_V[1,0:4])*np.random.rand(npop,nvar)
OUT = evaluate(pop)

oldpop = pop.copy()
oldOUT = OUT.copy()


In [85]:
# bounds
low = min_v[0:4]
up  = max_v[0:4]
d = up - low

# ================= GA LOOP =================
for gen in range(gNr):

    # -------- tournament selection --------
    idx = np.random.randint(0, npop, (npop,2))

    fit = oldOUT[idx,1]     # compare Te

    win = np.where(fit[:,0] < fit[:,1], idx[:,0], idx[:,1])

    parent1 = oldpop[win]
    parent2 = oldpop[np.random.randint(0,npop,npop)]

    # -------- crossover --------
    mask = np.random.rand(npop,nvar) < crossprob
    coeff = np.random.rand(npop,nvar)

    child = parent1.copy()

    child[mask] = (
        parent1[mask]*(1-coeff[mask]) +
        parent2[mask]*coeff[mask]
    )

    # -------- mutation --------
    mutmask = np.random.rand(npop,nvar) < mutprob
    child += mutmask * (mut_scale * d * np.random.randn(npop,nvar))

    child = np.clip(child, low, up)

    pop = child

    # -------- evaluate --------
    OUT = evaluate(pop)

    # -------- replacement --------
    err = np.abs(OUT[:,1] - CTe[5])

    new_idx = np.argmin(err)
    old_idx = np.random.randint(npop)

    if np.abs(oldOUT[old_idx,1] - CTe[5]) > err[new_idx]:
        oldpop[old_idx] = pop[new_idx]
        oldOUT[old_idx] = OUT[new_idx]

    # -------- best --------
    idx_best = np.argmin(np.abs(oldOUT[:,1] - CTe[5]))

    OV = np.zeros(6)

    # denormalize input
    for k in range(4):
        OV[k] = oldpop[idx_best,k]

    # denormalize output
    OV[4] = oldOUT[idx_best,0]*(max_v[4]-min_v[4]) + min_v[4]
    OV[5] = oldOUT[idx_best,1]*(max_v[5]-min_v[5]) + min_v[5]

    if gen % 50 == 0:
        print(f"Gen {gen} → {OV}")

    # convergence
    if np.std(oldOUT[:,1]) < 1e-12:
        print("Converged at gen", gen)
        break


print("\nFINAL SOLUTION:")
print(OV)

Gen 0 → [3.39342380e+02 3.63521894e-01 7.21518816e+01 5.21115322e-01
 3.27953365e+01 7.09618566e+02]
Gen 50 → [3.48711208e+02 3.10051031e-01 6.73433628e+01 5.32133576e-01
 3.20201384e+01 7.00022662e+02]
Gen 100 → [3.52859328e+02 3.05753567e-01 6.83923149e+01 5.30174544e-01
 3.20398363e+01 7.00000037e+02]
Gen 150 → [3.54513089e+02 3.06278998e-01 6.89076647e+01 5.31746376e-01
 3.20458489e+01 7.00000003e+02]
Gen 200 → [3.54512554e+02 3.06245955e-01 6.89100438e+01 5.31736211e-01
 3.20459090e+01 7.00000000e+02]
Gen 250 → [3.54512491e+02 3.06246163e-01 6.89100062e+01 5.31736217e-01
 3.20459083e+01 7.00000000e+02]
Gen 300 → [3.54512491e+02 3.06246163e-01 6.89100062e+01 5.31736217e-01
 3.20459083e+01 7.00000000e+02]
Gen 350 → [3.54512491e+02 3.06246163e-01 6.89100062e+01 5.31736217e-01
 3.20459083e+01 7.00000000e+02]
Gen 400 → [3.54512554e+02 3.06245955e-01 6.89100436e+01 5.31736211e-01
 3.20459090e+01 7.00000000e+02]
Gen 450 → [3.54512554e+02 3.06245955e-01 6.89100438e+01 5.31736212e-01
 3.20

In [80]:
oldOUT

array([[0.86256972, 0.66007021],
       [0.46400032, 0.46694502],
       [0.78863705, 0.73616234],
       [0.70499416, 0.71062178],
       [0.57504604, 0.45915153]])

In [ ]:
# ---------------- NN forward (VECTOR) ----------------
def evaluate(pop):
    x = (pop - range_V[1,0:4]) / (range_V[0,0:4]-range_V[1,0:4])
    X=X = np.hstack((x, np.ones((x.shape[0], 1))))
    h = logistic(X @ W1)
    H = np.hstack((h, np.ones((h.shape[0],1))))
    return logistic(H @ W2)

In [ ]:
OUT

array([[0.82783519, 0.89022376],
       [0.30630985, 0.29383303],
       [0.76525297, 0.82839477],
       [0.92087304, 0.96079312],
       [0.55402137, 0.73360561]])

In [ ]:
# ---------------- INITIAL POP ----------------
pop = range_V[1,0:4] + (range_V[0,0:4]-range_V[1,0:4])*np.random.rand(npop,nvar)

OUT = evaluate(pop)

oldpop = pop.copy()
oldOUT = OUT.copy()

NameError: name 'evaluate' is not defined

In [ ]:
CTe

array([1.        , 1.        , 1.        , 1.        , 1.        ,
       0.33848872])

In [ ]:
range_V

array([[3.600000e+02, 5.000000e-01, 1.280000e+02, 5.500000e-01,
        5.042100e+01, 9.959671e+02],
       [1.800000e+02, 3.000000e-01, 6.400000e+01, 3.500000e-01,
        3.178824e+01, 6.982139e+02]])

In [ ]:
np.load("/content/drive/MyDrive/W1W2.npz")

NpzFile '/content/drive/MyDrive/W1W2.npz' with keys: W1, W2, max_v, min_v